** Note that some required dependencies may not be included in this archive file. 

In [ ]:
from utils import *
import cv2
import matplotlib.pyplot as plt
import numpy as np

### COLOURING EMPTY SPACE BLACK

In [ ]:
def color_empty_space_black(
    image,
    edge_rows,
    background_value=0,
    edge_padding=4,
    split_row=None,
    show_transformation=False
):
    """
    Color unnecessary outer space black, including the detected
    horizontal boundary and several neighboring rows.

    Parameters:
    * image: binary or grayscale uint8 image
    * edge_rows: detected horizontal edge row indices
    * background_value: value used for removed areas, normally 0
    * edge_padding: extra rows removed around each detected edge
    * split_row: row dividing top and bottom regions. If None,
      uses half the actual image height.
    * show_transformation: display before/after comparison

    Returns:
    * cleaned image
    """
    image = correct_image_type(image)
    cleaned_image = image.copy()

    height = cleaned_image.shape[0]

    if split_row is None:
        split_row = height // 2

    for row in edge_rows:
        if row < split_row:
            # Include the detected edge and rows just below it.
            cleanup_end = min(
                height,
                row + edge_padding + 1
            )

            cleaned_image[:cleanup_end, :] = background_value

        else:
            # Include the detected edge and rows just above it.
            cleanup_start = max(
                0,
                row - edge_padding
            )

            cleaned_image[cleanup_start:, :] = background_value
    
    if show_transformation:
        show_image_comparison(
            image,
            cleaned_image,
            title1="Before Empty-Space Cleanup",
            title2=(
                f"After Cleanup | Edges: {edge_rows} | "
                f"Padding: {edge_padding}"
            )
        )

    return cleaned_image

### TESTING DIFFERENT IMAGE BINARIZATION TECHNIQUES

In [ ]:
def binarize_image_tests(
    image,
    method="otsu",
    show_transformation=False
):
    """
    Binarize an image using different methods.

    Parameters
    ----------
    image : uint8 grayscale image

    method :
        "otsu"
        "clahe_otsu"
        "tophat_otsu"
        "background_subtract"
        "multi_otsu"
        "percentile"
        "li"
        "yen"
        "triangle"
        "isodata"
        "sauvola"
        "niblack"
        "local_otsu"
        "manual"

    Returns
    -------
    binary_img
    threshold_used
    """

    image = correct_image_type(image)
    image = cv2.GaussianBlur(
        image,
        (5, 5),
        0
        )
    
    clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8, 8)
        )

    image = clahe.apply(image)

    threshold_used = None

    #################################################
    # STANDARD OTSU
    #################################################

    if method == "otsu":

        threshold_used, binary_img = cv2.threshold(
            image,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

    #################################################
    # CLAHE + OTSU
    #################################################

    elif method == "clahe_otsu":

        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8, 8)
        )

        enhanced = clahe.apply(image)

        threshold_used, binary_img = cv2.threshold(
            enhanced,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

    #################################################
    # TOP-HAT + OTSU
    #################################################

    elif method == "tophat_otsu":

        kernel = cv2.getStructuringElement(
            cv2.MORPH_RECT,
            (9, 9)
        )

        enhanced = cv2.morphologyEx(
            image,
            cv2.MORPH_TOPHAT,
            kernel
        )

        threshold_used, binary_img = cv2.threshold(
            enhanced,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

    #################################################
    # BACKGROUND SUBTRACTION + OTSU
    #################################################

    elif method == "background_subtract":

        background = cv2.GaussianBlur(
            image,
            (51, 51),
            0
        )

        enhanced = cv2.subtract(
            image,
            background
        )

        enhanced = cv2.normalize(
            enhanced,
            None,
            0,
            255,
            cv2.NORM_MINMAX
        )

        threshold_used, binary_img = cv2.threshold(
            enhanced,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

    #################################################
    # MULTI OTSU
    #################################################

    elif method == "multi_otsu":

        from skimage.filters import threshold_multiotsu
        

        thresholds = threshold_multiotsu(
            image,
            classes=3
        )

        threshold_used = thresholds[1]

        binary_img = np.where(
            image > threshold_used,
            255,
            0
        ).astype(np.uint8)

        
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2,2)
        )

        binary_img = cv2.erode(
            binary_img,
            kernel,
            iterations=1
        )

    #################################################
    # LI THRESHOLD
    #################################################

    elif method == "li":

        from skimage.filters import threshold_li

        threshold_used = threshold_li(image)

        binary_img = np.where(
            image > threshold_used,
            255,
            0
        ).astype(np.uint8)

        print(
            f"Li Threshold: {threshold_used:.1f}"
        )

    #################################################
    # YEN THRESHOLD
    #################################################

    elif method == "yen":

        from skimage.filters import threshold_yen

        threshold_used = threshold_yen(image)

        binary_img = np.where(
            image > threshold_used,
            255,
            0
        ).astype(np.uint8)

        print(
            f"Yen Threshold: {threshold_used:.1f}"
        )

    #################################################
    # TRIANGLE THRESHOLD
    #################################################

    elif method == "triangle":

        from skimage.filters import threshold_triangle

        threshold_used = threshold_triangle(image)

        binary_img = np.where(
            image > threshold_used,
            255,
            0
        ).astype(np.uint8)

        print(
            f"Triangle Threshold: {threshold_used:.1f}"
        )

    ##################################################
    # ISODATA THRESHOLD
    ##################################################

    elif method == "isodata":

        from skimage.filters import threshold_isodata

        threshold_used = threshold_isodata(image)

        binary_img = np.where(
            image > threshold_used,
            255,
            0
        ).astype(np.uint8)

        print(
            f"Isodata Threshold: {threshold_used:.1f}"
        )

    ##################################################
    # SAUVOLA THRESHOLD
    ##################################################
    elif method == "sauvola":

        from skimage.filters import threshold_sauvola

        threshold_map = threshold_sauvola(
            image,
            window_size=25,
            k=0.15
        )

        threshold_used = "local"

        binary_img = np.where(
            image > threshold_map,
            255,
            0
        ).astype(np.uint8)

        print("Sauvola Threshold")

    ##################################################
    # NIBLACK THRESHOLD
    ##################################################

    elif method == "niblack":

        from skimage.filters import threshold_niblack

        threshold_map = threshold_niblack(
            image,
            window_size=25,
            k=0.1
        )

        threshold_used = "local"

        binary_img = np.where(
            image > threshold_map,
            255,
            0
        ).astype(np.uint8)

        print("Niblack Threshold")

    ##################################################
    # LOCAL OTSU THRESHOLD
    ##################################################
    elif method == "local_otsu":

        from skimage.filters import rank
        from skimage.morphology import disk

        local_thresholds = rank.otsu(
            image,
            disk(15)
        )

        threshold_used = "local"

        binary_img = np.where(
            image >= local_thresholds,
            255,
            0
        ).astype(np.uint8)

        print("Local Otsu")


    #################################################
    # PERCENTILE THRESHOLD
    #################################################

    elif method == "percentile":

        threshold_used = np.percentile(
            image,
            85
        )

        binary_img = np.where(
            image > threshold_used,
            255,
            0
        ).astype(np.uint8)

    #################################################
    # MANUAL THRESHOLD
    #################################################
    elif method == "manual":
        # low-ish threshold
        threshold = 120

        binary_img = np.where(
            image > threshold,
            255,
            0
        ).astype(np.uint8)

        # thin slightly afterward
        kernel = cv2.getStructuringElement(
            cv2.MORPH_CROSS,
            (3,3)
        )

        binary_img = cv2.erode(
            binary_img,
            kernel,
            iterations=1
        )

    #################################################

    else:
        raise ValueError(
            f"Unknown method: {method}"
        )

    if show_transformation:

        show_image_comparison(
            image,
            binary_img,
            title1="Original",
            title2=f"{method} | T={threshold_used}"
        )

    print(
        f"Method={method}, "
        f"Threshold={threshold_used}"
    )

    return binary_img, threshold_used